# 03 · Option 2 · Microsoft Foundry Evaluations

The harness still drives Copilot Chat and gathers the evidence (Foundry can't call Copilot), then hands each ask to
a **Foundry evaluation run**:

* code evaluators: the same value, artifact and semantic-model scorers as Option 1, registered in Foundry;
* prompt evaluators: answer correctness and consistency of Copilot's explanation (1 to 5);
* built-in evaluators: coherence and similarity to the reference answer.

You get Foundry's report and run comparison, and the results are written back to the Eventhouse so the scorecard
app shows both options side by side when the service succeeds.

**Optional; not the Friday default.** Preview dataset runs can fail before grades/error detail. Preserve original
service IDs/status and stop, with no local-grade fallback or blind new dataset run. No automatic jobs.

In [ ]:
# Make the cfeval package available.
# - Local clone (VS Code / Jupyter): the cell below finds src/ in the repo.
# - Fabric notebook: uncomment the %pip line (or attach an Environment with the package).
# %pip install -q "cfeval[copilot,kusto,foundry] @ git+https://github.com/bgbailey/copilot_iq_eval"
import sys, pathlib
repo = pathlib.Path.cwd()
while repo != repo.parent and not (repo / "src" / "cfeval").exists():
    repo = repo.parent
if (repo / "src" / "cfeval").exists() and str(repo / "src") not in sys.path:
    sys.path.insert(0, str(repo / "src"))
import cfeval
print("cfeval", cfeval.__version__)

In [ ]:
import os

interactive_login = False  # True only for an explicit manual sign-in; reset afterwards.
run_live = False  # Separate approval required for Copilot AND a Foundry dataset evaluation.
outcome = None

os.environ.update({
    # Copilot Chat API: app registration + Copilot-licensed test user (docs/SETUP.md)
    "TENANT_ID": "<tenant id>",
    "TENANT_LABEL": "<tenant name / domain>",
    "OPERATOR_UPN": "eval.operator@<your-domain>",
    "AUTH_RUN_LABEL": "cfeval-option2-login",
    "CLIENT_ID": "<app registration client id>",
    "TEST_USER_UPN": "eval.tester@<your-domain>",
    "TOKEN_CACHE": "keyvault",                      # optional explicit persistence
    "KEY_VAULT_URL": "https://<your-vault>.vault.azure.net/",
    # Golden set: upload your CSV to the lakehouse Files area
    "GOLDEN_SET": "/lakehouse/default/Files/golden_questions.csv",
    # Telemetry Eventhouse
    "EVENTHOUSE_QUERY_URI": "https://<your-eventhouse>.z0.kusto.fabric.microsoft.com",
    "EVENTHOUSE_DATABASE": "<actual engine databaseName, often the KQL database GUID>",
    # DAX capture: workspace monitoring of the semantic-model workspace
    "MONITORING_QUERY_URI": "https://<monitoring-eventhouse>.z0.kusto.fabric.microsoft.com",
    "MONITORING_DATABASE": "<monitoring KQL database name>",
    "SEMANTIC_MODEL_WORKSPACE_ID": "<workspace id>",
    # Foundry project; in Fabric use a service principal whose secret is in Key Vault
    "FOUNDRY_PROJECT_ENDPOINT": "https://<account>.services.ai.azure.com/api/projects/<project>",
    "FOUNDRY_EVAL_DEPLOYMENT": "gpt-5-mini",
    "FOUNDRY_SP_CLIENT_ID": "<service principal client id>",
    "FOUNDRY_SP_SECRET_NAME": "foundry-sp-secret",
})
from cfeval.config import Settings
settings = Settings.load(env_file=None)

In [ ]:
from cfeval.auth import DelegatedAuth, notebook_login_callback

if interactive_login or run_live:
    callback = notebook_login_callback(settings) if interactive_login else None
    auth = DelegatedAuth.from_settings(settings, login_callback=callback)
    auth.token(interactive=interactive_login)
    print("Verified test user", auth.signed_in_upn(), "tenant", settings.tenant_id)
else:
    print("No authentication or cloud calls (live/login flags false).")

## Run with Foundry as the evaluation framework
Only enable for a separately approved dataset run. Default acceptance is Option 1 through the local scripted route.

In [ ]:
from cfeval.cli import build_harness

if run_live:
    outcome = build_harness(settings, option="foundry", run_name="Optional Option 2",
                            repeats=1, explain_mode="followup", chat_max_retries=0).run()
    print("Foundry operation:", outcome.foundry)
    if outcome.status in ("partial", "failed"):
        raise RuntimeError("Foundry backend ERROR; preserve operation IDs and stop, no fallback/retry")
    print(outcome.metrics)
else:
    print("No Copilot or Foundry dataset requests (run_live=False).")

In [ ]:
from pprint import pprint

cols = ["QuestionId", "Verdict", "ValueScore", "ArtifactScore", "ModelScore",
        "JudgeAnswerScore", "WhyScore", "Reasons", "JudgeNotes"]
if outcome is not None:
    pprint([{key: row[key] for key in cols} for row in outcome.results])

## What Foundry stores
* Evaluators `cfeval_value_match`, `cfeval_artifact_match`, `cfeval_model_match` (code) and
  `cfeval_answer_correctness`, `cfeval_why_consistency` (prompt), versioned when their definition changes.
* One evaluation per run with its report URL (also saved in `EvalRun.FoundryReportUrl`).